# 26 - Regularization: the theory

**Section:** Regularization | **Prereqs:** `Cross Validation/DataLoader.ipynb` | **Next:** `Regularization/Mini_batch.ipynb`

Regularisation is anything that makes the training task *harder* in order to make
the resulting model *simpler*. Simpler models generalise. The reference notebook
for the whole section.

**The three families used in this course**

| Method | What it does | Where it lives |
|---|---|---|
| **Dropout** | randomly zeroes activations during training | a layer in `forward` |
| **L2 / weight decay** | penalises large weights, shrinking all of them | `weight_decay=` in the optimizer |
| **L1** | penalises absolute weights, driving many to exactly 0 | added to the loss by hand |
| **Mini-batching** | noisy gradient estimates | the DataLoader's `batch_size` |

**`model.train()` vs `model.eval()`** - the switch you must not forget. Dropout
and batch-norm behave *differently* in the two modes. Evaluating without
`.eval()` gives noisy, pessimistic numbers; training without `.train()` after an
eval silently disables your regularisation.

**`torch.no_grad()`** is a separate concern: it stops autograd from building a
graph. It saves memory and time during evaluation but changes no layer's
behaviour. You usually want both.

**The scaling detail** in the markdown below is worth understanding: dropout
divides surviving activations by `1-p` during training, so the expected sum
stays constant and no rescaling is needed at test time.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader , TensorDataset
import torch.nn.functional as F


from sklearn.model_selection import train_test_split
%matplotlib inline

## train() and eval() modes

In [ ]:
# A model built only to demonstrate the mode switch.
# .train() and .eval() flip a boolean on every submodule; they do NOT train or
# evaluate anything by themselves.
# torch.no_grad() must be used as a CONTEXT MANAGER - `with torch.no_grad():`.
# Calling it bare creates the object and throws it away, disabling nothing.
# nn.Softmax needs an explicit dim; and note a model ending in Softmax pairs with
# NLLLoss-style losses only, never with CrossEntropyLoss.
model = nn.Sequential(
    nn.Linear(2,20),
    nn.ReLU(),
    nn.Linear(20,20),
    nn.ReLU(),
    nn.Linear(20,20),
    nn.ReLU(),
    nn.Linear(20,2),
    nn.Softmax(dim=1)
)



model.train() # Regrualrization is on, active by default

model.eval() # Regularization is off

with torch.no_grad():   # Stops building the autograd graph - wrap evaluation in this
  pass


# training loop :
#    model.train()

#    TRAINING
#    batch loop:


#    CALCULATING ACCURACY
#    model.eval()
#    with torch.no_grad(), only for really large models
#    compute accuracy





In [ ]:
# .grad is None before any backward() call - gradients do not exist until the
# first backward pass. A None here after training means the parameter is
# disconnected from the loss.
for i in model.parameters():
  print(i.grad)
  print()

None

None

None

None

None

None

None

None



In [ ]:
# Weight shapes read (out_features, in_features); biases are 1-D.
for i in model.parameters():
 print(i.shape)

torch.Size([20, 2])
torch.Size([20])
torch.Size([20, 20])
torch.Size([20])
torch.Size([20, 20])
torch.Size([20])
torch.Size([2, 20])
torch.Size([2])


In [ ]:
model

Sequential(
  (0): Linear(in_features=2, out_features=20, bias=True)
  (1): ReLU()
  (2): Linear(in_features=20, out_features=20, bias=True)
  (3): ReLU()
  (4): Linear(in_features=20, out_features=20, bias=True)
  (5): ReLU()
  (6): Linear(in_features=20, out_features=2, bias=True)
  (7): Softmax(dim=None)
)

# Dropout Regularization


## Dropout scaling in PyTorch

During **training**, `nn.Dropout(p)` keeps each activation with probability \(1-p\) and scales the survivors by  

$$
q = \frac{1}{1-p}.
$$  

Hence an activation (or weight) \(w\) becomes  

$$
w_{\text{train}} = w \, q = \frac{w}{1-p}.
$$  

When you switch to **evaluation** mode with `model.eval()`, dropout is disabled and **no scaling** is applied.


---


In [ ]:
# A vector of ones, so the effect of dropout is visible as pure arithmetic.
tensor = torch.ones(10)
tensor

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [ ]:
# Dropout with p=0.75 in TRAINING mode: about 75% of entries become 0, and the
# survivors become 1/(1-p) = 4, not 1.
# That rescaling keeps the expected sum unchanged, which is why no correction is
# needed at test time. Run it repeatedly - a different mask each call.
p = 0.75

dropout = nn.Dropout(p=p)

dropout.train()

dropout(tensor)




tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [ ]:
# The same layer in EVAL mode: identity. Nothing dropped, nothing scaled.
# This is why forgetting .eval() ruins your test numbers.
dropout.eval()

dropout(tensor)

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [ ]:
# The functional form F.dropout does NOT read the module's mode - it defaults to
# training=True and drops even during evaluation. That is the classic silent bug.
# Inside an nn.Module, pass training=self.training explicitly.
## Using functional dropout

y = F.dropout(tensor,p=0.5)

## Is not effected by .eval(), So

z = F.dropout(tensor, p=0.5, training=False)


In [ ]:
# y was dropped (functional default), z was not (training=False). Same call,
# different behaviour - hence the preference for nn.Dropout as a layer.
y , z

(tensor([0., 0., 2., 0., 2., 0., 2., 2., 2., 2.]),
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]))

## L1 vs L2 Regularization — Definitions & Formulas

|                 | **L1 (Lasso)**                                             | **L2 (Ridge / weight-decay)**                   |
|-----------------|------------------------------------------------------------|-------------------------------------------------|
| **Penalty term**| $\displaystyle \lambda \sum_{i=1}^{n} \lvert w_i \rvert$  | $\displaystyle \lambda \sum_{i=1}^{n} w_i^{2}$ |
| **Vector form** | $\lambda \lVert \mathbf{w} \rVert_{1}$                     | $\lambda \lVert \mathbf{w} \rVert_{2}^{\,2}$   |
| **Gradient**    | $\lambda\,\mathrm{sign}(w_i)$ (0 when $w_i\!=\!0$)         | $2\lambda\,w_i$                                |
| **Effect**      | Drives many weights **exactly to 0 → sparsity**            | Smoothly shrinks all weights → **smaller**, rarely 0 |
| **Typical use** | Feature selection, compressed models                       | Stabilising training, reducing over-fitting    |

Total loss with data term $\mathcal{L}_{\text{data}}$:

$$
\mathcal{L}_{\text{total}}^{\text{L1}}
  = \mathcal{L}_{\text{data}} + \lambda \| \mathbf{w} \|_{1}
\qquad
\mathcal{L}_{\text{total}}^{\text{L2}}
  = \mathcal{L}_{\text{data}} + \lambda \| \mathbf{w} \|_{2}^{\,2}
$$
